# Generative AI: Assignment 1

This notebook covers:
- **Part 1:** Topic Detection & Summarization of BBC News Articles 
- **Part 2:** Job Postings Analysis - Role Categorization & Requirements Extraction 

## Initial Setup
### Set API key for Groq
Click [here](https://console.groq.com/keys) to create an API key for Groq, if not already created.

In [1]:
import os, json, re, getpass
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [8]:
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0)

---
# Part 1: Topic Detection and Summarization of News Articles


## Step 1: Load the Dataset
Using the BBC News Full-Text dataset, limited to the first 30 articles.

In [9]:
news_df = pd.read_csv("bbc-news-data.csv", sep="\t")
news_df = news_df.head(30).reset_index(drop=True)
print(news_df.shape)
news_df.head()

(30, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


## Step 2: Define the Topic Classification Task 

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

topic_categories = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

classification_template = ChatPromptTemplate([
    ("system", "You are a news editor who classifies articles into a single topic category."),
    ("human", """Analyze the following news article and identify its topic as one of the following categories: {categories}.

Examples:
Article: "The prime minister announced new legislation in parliament today..." -> Politics
Article: "The striker scored a hat-trick to win the match for his team..." -> Sport

Article:
{article}

Return ONLY the single category label, nothing else."""),
])

classification_chain = classification_template | llm | StrOutputParser()

In [11]:
# Show this works for a sample datapoint
sample_article = news_df.loc[0, "content"]

sample_topic = classification_chain.invoke({
    "categories": ", ".join(topic_categories),
    "article": sample_article
}).strip()

print("Predicted Topic:", sample_topic)
print("Actual Category:", news_df.loc[0, "category"])

Predicted Topic: Business
Actual Category: business


## Step 3: Define the Summarization Task

In [12]:
summarization_template = ChatPromptTemplate([
    ("system", "You are a skilled news summarizer."),
    ("human", """Summarize the main points of the following news article in 2-3 sentences.
Capture the who/what/when/where/why as applicable, without adding personal commentary.

Article:
{article}"""),
])

summarization_chain = summarization_template | llm | StrOutputParser()

In [13]:
# Show this works for a sample datapoint
sample_summary = summarization_chain.invoke({"article": sample_article}).strip()
print(sample_summary)

TimeWarner reported a 76% jump in quarterly profit to $1.13 billion for the three months ending December, driven by higher sales of high‑speed internet connections, increased advertising revenue and one‑off gains that offset a dip at Warner Bros and a loss of AOL subscribers. The company, now holding an 8% stake in Google, also announced it must restate its 2000 and 2003 results after an SEC probe into AOL, while its film division saw profits fall 27% due to box‑office flops. For the full year, TimeWarner posted a 27% rise in profit to $3.36 billion and projected about 5% operating‑earnings growth for 2005.


## Step 4: Key Entity Extraction


In [14]:
from pydantic import BaseModel, Field
from typing import List

class KeyEntities(BaseModel):
    """Important entities mentioned in a news article."""
    people: List[str] = Field(description="Notable people mentioned in the article")
    organizations: List[str] = Field(description="Organizations/companies mentioned in the article")
    locations: List[str] = Field(description="Places/locations mentioned in the article")

entity_template = ChatPromptTemplate([
    ("system", "You extract key named entities from news articles."),
    ("human", """From the article below, list the names of any important people, organizations, or places mentioned.

Article:
{article}"""),
])

entity_chain = entity_template | llm.with_structured_output(KeyEntities)

In [15]:
# Show this works for a sample datapoint
sample_entities = entity_chain.invoke({"article": sample_article})
sample_entities

KeyEntities(people=['Richard Parsons'], organizations=['TimeWarner', 'Google', 'AOL', 'Warner Bros', 'Securities and Exchange Commission', 'SEC', 'Bertelsmann', 'AOL Europe'], locations=['United States', 'Germany'])

## Step 5: Update the DataFrame with Results
Combine all three tasks into a single structured chain (fewer LLM calls, more efficient) and apply it across the first 30 articles.

In [16]:
class ArticleAnalysis(BaseModel):
    """Structured analysis of a news article: topic, summary and key entities."""
    Detected_Topic: str = Field(description=f"The single best-fit topic, one of: {', '.join(topic_categories)}")
    Summary: str = Field(description="A concise 2-3 sentence summary of the article")
    Key_Entities: List[str] = Field(description="Important people, organizations, or locations mentioned in the article")

analysis_template = ChatPromptTemplate([
    ("system", "You are an expert news analyst."),
    ("human", """Analyze the following news article and provide:
1. Its topic - one of: {categories}
2. A 2-3 sentence summary capturing the key points
3. A list of key entities (notable people, organizations, or locations)

Article:
{article}"""),
])

analysis_chain = analysis_template | llm.with_structured_output(ArticleAnalysis)

In [17]:
results = []
for idx, row in news_df.iterrows():
    try:
        analysis = analysis_chain.invoke({
            "categories": ", ".join(topic_categories),
            "article": row["content"]
        })
        results.append(analysis.model_dump())
    except Exception as e:
        print(f"Row {idx} failed: {e}")
        results.append({"Detected_Topic": "Not specified", "Summary": "Not specified", "Key_Entities": []})

results_df = pd.DataFrame(results)
results_df.head()

,Detected_Topic,Summary,Key_Entities
0,Business,TimeWarner reported a 76% jump in quarterly pr...,"[TimeWarner, Google, AOL, Warner Bros, Richard..."
1,Business,The dollar rose to its highest level against t...,"[Federal Reserve, Alan Greenspan, Bank of Amer..."
2,Business,"Menatep Group, the owner of the former Yukos p...","[Yukos, Menatep Group, Rosneft, Yugansk, Jamie..."
3,Business,British Airways reported a 40% drop in pre‑tax...,"[British Airways, Rod Eddington, Martin Brough..."
4,Business,Shares in UK drinks and food company Allied Do...,"[Allied Domecq, Pernod Ricard, Wall Street Jou..."


In [18]:
# Final merged dataframe with all original and new columns together
news_final_df = pd.concat([news_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)
news_final_df.head()

,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,Business,TimeWarner reported a 76% jump in quarterly pr...,"[TimeWarner, Google, AOL, Warner Bros, Richard..."
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,Business,The dollar rose to its highest level against t...,"[Federal Reserve, Alan Greenspan, Bank of Amer..."
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,Business,"Menatep Group, the owner of the former Yukos p...","[Yukos, Menatep Group, Rosneft, Yugansk, Jamie..."
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,Business,British Airways reported a 40% drop in pre‑tax...,"[British Airways, Rod Eddington, Martin Brough..."
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,Business,Shares in UK drinks and food company Allied Do...,"[Allied Domecq, Pernod Ricard, Wall Street Jou..."


In [19]:
news_final_df.to_csv("part1_news_analysis_results.csv", index=False)
news_final_df.shape

(30, 7)

---
# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction


## Step 1: Load the Dataset
Using the job postings dataset, limited to the first 25 postings.

In [20]:
jobs_df = pd.read_csv("job_title_des.csv")
jobs_df = jobs_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})
jobs_df = jobs_df[["Job_Title", "Job_Description"]].head(25).reset_index(drop=True)
print(jobs_df.shape)
jobs_df.head()

(25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2: Define the Job Category Classification Task

In [21]:
job_categories = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Sales", "Operations", "Other"]

job_classification_template = ChatPromptTemplate([
    ("system", "You are an HR specialist who categorizes job postings into a broad domain."),
    ("human", """Given the following job title and description, categorize the job into one of the following domains: {categories}.
If unsure, use "Other".

Job: {job_title}
Description: {job_description}

Return ONLY the single domain category label, nothing else."""),
])

job_classification_chain = job_classification_template | llm | StrOutputParser()

In [22]:
# Show this works for a sample datapoint
sample_job_title = jobs_df.loc[0, "Job_Title"]
sample_job_desc = jobs_df.loc[0, "Job_Description"]

sample_category = job_classification_chain.invoke({
    "categories": ", ".join(job_categories),
    "job_title": sample_job_title,
    "job_description": sample_job_desc
}).strip()

print("Job Title:", sample_job_title)
print("Predicted Category:", sample_category)

Job Title: Flutter Developer
Predicted Category: Technology/IT


## Step 3: Define the Requirements Extraction Task

In [23]:
class JobRequirements(BaseModel):
    """Key requirements extracted from a job description."""
    Required_Skills: List[str] = Field(description="Key skills, programming languages, tools or domain knowledge mentioned. Empty list if none.")
    Education_Required: str = Field(description="Minimum education level required/preferred (e.g. Bachelor's, MBA). 'Not specified' if not mentioned.")
    Experience_Required: str = Field(description="Years of experience or experience level required (e.g. '3+ years'). 'Not specified' if not mentioned.")

requirements_template = ChatPromptTemplate([
    ("system", "You extract structured hiring requirements from job descriptions."),
    ("human", """Extract the required skills, education level, and years of experience from the job description below.
If a field is not mentioned, use "Not specified".

Job Title: {job_title}
Job Description: {job_description}"""),
])

requirements_chain = requirements_template | llm.with_structured_output(JobRequirements)

In [24]:
# Show this works for a sample datapoint
sample_requirements = requirements_chain.invoke({
    "job_title": sample_job_title,
    "job_description": sample_job_desc
})
sample_requirements

JobRequirements(Required_Skills=['Flutter'], Education_Required='Not specified', Experience_Required='1 year (Preferred)')

## Step 4 & 5: Apply the LLM Chain to Each Job Posting and Update the DataFrame

In [25]:
job_results = []
for idx, row in jobs_df.iterrows():
    try:
        category = job_classification_chain.invoke({
            "categories": ", ".join(job_categories),
            "job_title": row["Job_Title"],
            "job_description": row["Job_Description"]
        }).strip()

        requirements = requirements_chain.invoke({
            "job_title": row["Job_Title"],
            "job_description": row["Job_Description"]
        })

        job_results.append({
            "Predicted_Category": category,
            "Required_Skills": requirements.Required_Skills,
            "Education_Required": requirements.Education_Required,
            "Experience_Required": requirements.Experience_Required,
        })
    except Exception as e:
        print(f"Row {idx} failed: {e}")
        job_results.append({
            "Predicted_Category": "Not specified",
            "Required_Skills": [],
            "Education_Required": "Not specified",
            "Experience_Required": "Not specified",
        })

job_results_df = pd.DataFrame(job_results)
job_results_df.head()

,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Technology/IT,"[Python, Django, Flask, REST API development, ...",Not specified,Not specified
2,Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",3+ years
3,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Not specified
4,Technology/IT,"[React, React Native, JavaScript, HTML, CSS, R...",Computer Science or equivalent,"5+ years web development, 2+ years recent Reac..."


In [26]:
# Final merged dataframe with all original and new columns together
jobs_final_df = pd.concat([jobs_df.reset_index(drop=True), job_results_df.reset_index(drop=True)], axis=1)
jobs_final_df.head()

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, Flask, REST API development, ...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",3+ years
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, React Native, JavaScript, HTML, CSS, R...",Computer Science or equivalent,"5+ years web development, 2+ years recent Reac..."


In [27]:
jobs_final_df.to_csv("part2_job_analysis_results.csv", index=False)
jobs_final_df.shape

(25, 6)

---
## Notes
- Both parts use `llm.with_structured_output(...)` with a Pydantic schema so the model's output is validated/parsed automatically (same pattern as `3. Structured Output Generation.ipynb`).
- Each row is wrapped in a `try/except` so a single rate-limited/failed call doesn't stop the whole loop - failed rows fall back to `"Not specified"`.